[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/slam-handbook-python/blob/main/part1_foundations/ch01_factor_graphs/02_nonlinear_slam.ipynb)

# Chapter 1: 非線形SLAMとGauss-Newton法

**SLAM Handbook Section 1.3–1.4** の内容を実装します。

## このNotebookで学ぶこと

1. **2D SLAM問題** — ポーズ $(x, y, \theta)$ とランドマーク $(x, y)$ の推定
2. **非線形計測関数** — bearing観測 $h(\mathbf{p}, \boldsymbol{\ell}) = \text{atan2}(\ell_y - p_y, \ell_x - p_x) - \theta$
3. **Gauss-Newton法のスクラッチ実装** — 線形化 → 正規方程式 → 更新の反復
4. **Levenberg-Marquardt法** — ダンピングの効果
5. **収束の可視化** — 各反復のコスト減少と軌跡の変化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12

def normalize_angle(a):
    """角度を [-π, π] に正規化"""
    return (a + np.pi) % (2 * np.pi) - np.pi

## 2.1 問題設定: 2D Landmark SLAM

ロボットが2D平面上を移動しながらランドマークを観測します。

- **ポーズ**: $\mathbf{p}_i = (x_i, y_i, \theta_i) \in \mathbb{R}^2 \times S^1$（位置+向き）
- **ランドマーク**: $\boldsymbol{\ell}_j = (\ell_{x,j}, \ell_{y,j}) \in \mathbb{R}^2$
- **オドメトリ計測**: $\mathbf{o}_t = \mathbf{p}_{t+1} - \mathbf{p}_t + \boldsymbol{\eta}_{\text{odom}}$（相対移動量）
- **Bearing観測**: $z = \text{atan2}(\ell_y - p_y, \ell_x - p_x) - \theta + \eta_{\text{obs}}$（SLAM Handbook Eq. 1.13）

Bearing観測は **非線形** なので、Gauss-Newton法が必要です。

In [ ]:
# =============================================================
# Ground Truth の生成
# =============================================================
np.random.seed(123)

# ロボットの真のポーズ: (x, y, theta) — 三角形を描く軌跡
poses_true = np.array([
    [0.0, 0.0, 0.0],        # p0: 原点、東向き
    [2.0, 0.0, np.pi/3],    # p1: 東に移動、少し左向き
    [3.0, 1.7, 2*np.pi/3],  # p2: 北東に移動
    [2.0, 3.4, np.pi],      # p3: 北に移動、西向き
    [0.0, 3.4, -2*np.pi/3], # p4: 西に移動
    [-1.0, 1.7, -np.pi/3],  # p5: 南西に移動
])
n_poses = len(poses_true)

# ランドマークの真の位置
landmarks_true = np.array([
    [4.0, 1.0],   # l0
    [4.0, 3.0],   # l1
    [-2.0, 1.0],  # l2
    [1.0, 5.0],   # l3
])
n_landmarks = len(landmarks_true)

# 可視化
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
# ポーズ
for i, (x, y, th) in enumerate(poses_true):
    ax.plot(x, y, 'bo', markersize=8)
    ax.arrow(x, y, 0.4*np.cos(th), 0.4*np.sin(th),
             head_width=0.12, head_length=0.08, fc='blue', ec='blue')
    ax.text(x+0.15, y+0.2, f'p{i}', fontsize=11, color='blue', fontweight='bold')
ax.plot(poses_true[:, 0], poses_true[:, 1], 'b--', alpha=0.3)

# ランドマーク
for j, (lx, ly) in enumerate(landmarks_true):
    ax.plot(lx, ly, 'r*', markersize=15)
    ax.text(lx+0.15, ly+0.15, f'l{j}', fontsize=11, color='red', fontweight='bold')

ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Ground Truth: 6 Poses + 4 Landmarks', fontsize=14, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout()
plt.show()

print(f"ポーズ数: {n_poses}, ランドマーク数: {n_landmarks}")
print(f"状態ベクトルの次元: {n_poses*3 + n_landmarks*2} "
      f"(poses: {n_poses}×3 = {n_poses*3}, landmarks: {n_landmarks}×2 = {n_landmarks*2})")

In [ ]:
# =============================================================
# ノイズ付き観測の生成
# =============================================================

# ノイズパラメータ
sigma_odom_xy = 0.1    # オドメトリの位置ノイズ (m)
sigma_odom_th = 0.05   # オドメトリの角度ノイズ (rad)
sigma_bearing = 0.05   # Bearing観測のノイズ (rad)

Sigma_odom = np.diag([sigma_odom_xy**2, sigma_odom_xy**2, sigma_odom_th**2])
Sigma_prior = np.diag([0.01**2, 0.01**2, 0.001**2])  # 最初のポーズのprior（とても強い）

# --- オドメトリ計測 ---
odom_measurements = []
for i in range(n_poses - 1):
    delta_true = poses_true[i+1] - poses_true[i]
    delta_true[2] = normalize_angle(delta_true[2])
    noise = np.random.multivariate_normal([0, 0, 0], Sigma_odom)
    odom_measurements.append(delta_true + noise)

# --- Bearing観測 ---
# どのポーズからどのランドマークが見えるか定義
visibility = [
    (0, 0), (0, 2),  # p0 → l0, l2
    (1, 0), (1, 1),  # p1 → l0, l1
    (2, 1), (2, 3),  # p2 → l1, l3
    (3, 3), (3, 2),  # p3 → l3, l2
    (4, 2), (4, 3),  # p4 → l2, l3
    (5, 2), (5, 0),  # p5 → l2, l0
]

bearing_measurements = []
for pose_idx, lm_idx in visibility:
    px, py, pth = poses_true[pose_idx]
    lx, ly = landmarks_true[lm_idx]
    bearing_true = normalize_angle(np.arctan2(ly - py, lx - px) - pth)
    noise = np.random.normal(0, sigma_bearing)
    bearing_measurements.append(bearing_true + noise)

print(f"オドメトリ計測数: {len(odom_measurements)}")
print(f"Bearing観測数: {len(bearing_measurements)}")
print(f"ファクター数合計: 1 (prior) + {len(odom_measurements)} (odom) + {len(bearing_measurements)} (bearing) "
      f"= {1 + len(odom_measurements) + len(bearing_measurements)}")

## 2.2 Gauss-Newton法の実装

### アルゴリズム (SLAM Handbook Section 1.4.2)

1. 初期推定値 $\mathbf{x}^0$ を設定
2. 以下を収束するまで繰り返す:
   - 各ファクターで残差 $\mathbf{e}_i = \mathbf{z}_i - \mathbf{h}_i(\mathbf{x}^t)$ とヤコビアン $\mathbf{H}_i$ を計算
   - Whitening: $\mathbf{A}_i = \boldsymbol{\Sigma}_i^{-1/2} \mathbf{H}_i$, $\mathbf{b}_i = \boldsymbol{\Sigma}_i^{-1/2} \mathbf{e}_i$
   - 正規方程式を解く: $(\mathbf{A}^\top\mathbf{A}) \boldsymbol{\delta} = \mathbf{A}^\top\mathbf{b}$
   - 状態を更新: $\mathbf{x}^{t+1} = \mathbf{x}^t + \boldsymbol{\delta}$

### ヤコビアンの導出

**Bearing観測** $h(\mathbf{p}, \boldsymbol{\ell}) = \text{atan2}(\ell_y - p_y, \ell_x - p_x) - \theta$ のヤコビアン:

$$\frac{\partial h}{\partial \mathbf{p}} = \begin{bmatrix} \frac{\ell_y - p_y}{d^2} & \frac{-(l_x - p_x)}{d^2} & -1 \end{bmatrix}, \quad
\frac{\partial h}{\partial \boldsymbol{\ell}} = \begin{bmatrix} \frac{-(l_y - p_y)}{d^2} & \frac{\ell_x - p_x}{d^2} \end{bmatrix}$$

ここで $d^2 = (\ell_x - p_x)^2 + (\ell_y - p_y)^2$

In [ ]:
# =============================================================
# 状態ベクトルのヘルパー関数
# =============================================================
# 状態ベクトル x の構成:
#   [p0_x, p0_y, p0_th, p1_x, p1_y, p1_th, ..., l0_x, l0_y, l1_x, l1_y, ...]
#   pose i → indices [3*i : 3*i+3]
#   landmark j → indices [3*n_poses + 2*j : 3*n_poses + 2*j + 2]

n_state = 3 * n_poses + 2 * n_landmarks  # 状態ベクトルの次元

def pose_idx(i):
    """ポーズ i の状態ベクトル中のインデックス範囲"""
    return slice(3*i, 3*i + 3)

def lm_idx(j):
    """ランドマーク j の状態ベクトル中のインデックス範囲"""
    start = 3 * n_poses + 2 * j
    return slice(start, start + 2)

def get_pose(x, i):
    return x[pose_idx(i)]

def get_landmark(x, j):
    return x[lm_idx(j)]

print(f"状態ベクトルの次元: {n_state}")
print(f"pose 0 → x[{pose_idx(0)}]")
print(f"pose 5 → x[{pose_idx(5)}]")
print(f"landmark 0 → x[{lm_idx(0)}]")
print(f"landmark 3 → x[{lm_idx(3)}]")

In [ ]:
# =============================================================
# Gauss-Newton法の実装
# =============================================================

def gauss_newton_slam(x0, max_iter=20, tol=1e-6, lm_lambda=0.0):
    """2D Landmark SLAMをGauss-Newton法（またはLM法）で解く
    
    Parameters
    ----------
    x0 : array, shape (n_state,) — 初期推定値
    max_iter : int — 最大反復回数
    tol : float — 収束判定しきい値 (||δ|| < tol)
    lm_lambda : float — LMダンピング (0ならGauss-Newton)
    
    Returns
    -------
    x : 最終推定値
    history : 各反復の (コスト, ||δ||) のリスト
    """
    x = x0.copy()
    history = []
    
    Sigma_prior_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(Sigma_prior)))
    Sigma_odom_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(Sigma_odom)))
    sigma_bearing_inv = 1.0 / sigma_bearing
    
    for iteration in range(max_iter):
        # ファクターの総数
        n_rows = 3 + 3*(n_poses-1) + len(visibility)  # prior + odom + bearing
        
        H = np.zeros((n_rows, n_state))  # ヤコビアン
        e = np.zeros(n_rows)              # 残差ベクトル
        row = 0
        
        # --- Factor 1: Prior on pose 0 ---
        p0 = get_pose(x, 0)
        residual = p0 - poses_true[0]  # prior = 真の初期位置
        residual[2] = normalize_angle(residual[2])
        e[row:row+3] = Sigma_prior_inv_sqrt @ residual
        H[row:row+3, pose_idx(0)] = Sigma_prior_inv_sqrt
        row += 3
        
        # --- Factor 2: Odometry ---
        for i in range(n_poses - 1):
            pi = get_pose(x, i)
            pi1 = get_pose(x, i+1)
            predicted = pi1 - pi
            predicted[2] = normalize_angle(predicted[2])
            residual = predicted - odom_measurements[i]
            residual[2] = normalize_angle(residual[2])
            
            e[row:row+3] = Sigma_odom_inv_sqrt @ residual
            H[row:row+3, pose_idx(i)]   = Sigma_odom_inv_sqrt @ (-np.eye(3))
            H[row:row+3, pose_idx(i+1)] = Sigma_odom_inv_sqrt @ np.eye(3)
            row += 3
        
        # --- Factor 3: Bearing observations ---
        for k, (pi_idx, lj_idx) in enumerate(visibility):
            px, py, pth = get_pose(x, pi_idx)
            lx, ly = get_landmark(x, lj_idx)
            
            dx = lx - px
            dy = ly - py
            d2 = dx**2 + dy**2
            
            predicted_bearing = normalize_angle(np.arctan2(dy, dx) - pth)
            residual = normalize_angle(bearing_measurements[k] - predicted_bearing)
            
            e[row] = sigma_bearing_inv * residual
            
            # ヤコビアン ∂h/∂p = [dy/d², -dx/d², -1]
            J_pose = np.array([dy/d2, -dx/d2, -1.0])
            # ヤコビアン ∂h/∂ℓ = [-dy/d², dx/d²]
            J_lm = np.array([-dy/d2, dx/d2])
            
            # 残差は z - h(x) なので、ヤコビアンは -∂h/∂x (に右辺に使う場合)
            # ここではe = z - h(x)に対する ∂e/∂x = -∂h/∂x の形式
            # Gauss-Newton: H@δ ≈ e なので H = ∂e/∂x = -∂h/∂x
            H[row, pose_idx(pi_idx)] = sigma_bearing_inv * (-J_pose)
            H[row, lm_idx(lj_idx)]   = sigma_bearing_inv * (-J_lm)
            row += 1
        
        # コスト = ||e||²
        cost = 0.5 * np.dot(e, e)
        
        # 正規方程式: (H^T H + λI) δ = H^T e
        HTH = H.T @ H
        HTe = H.T @ e
        
        if lm_lambda > 0:
            HTH += lm_lambda * np.diag(np.diag(HTH))  # LMダンピング (Eq. 1.38)
        
        # 解く
        delta = np.linalg.solve(HTH, HTe)
        delta_norm = np.linalg.norm(delta)
        
        history.append((cost, delta_norm))
        
        # 更新
        x -= delta  # x^{t+1} = x^t - δ (残差 e = z - h なので符号注意)
        # 角度の正規化
        for i in range(n_poses):
            x[3*i + 2] = normalize_angle(x[3*i + 2])
        
        if delta_norm < tol:
            print(f"収束! (iteration {iteration+1}, ||δ|| = {delta_norm:.2e})")
            break
    else:
        print(f"最大反復回数 {max_iter} に到達 (||δ|| = {delta_norm:.2e})")
    
    return x, history

print("Gauss-Newton SLAM solver 定義完了")

In [ ]:
# =============================================================
# 初期推定値の作成（オドメトリの累積）
# =============================================================
x0 = np.zeros(n_state)

# ポーズの初期値: オドメトリを累積
x0[pose_idx(0)] = poses_true[0]  # 最初のポーズは既知
for i in range(n_poses - 1):
    prev = x0[pose_idx(i)]
    x0[pose_idx(i+1)] = prev + odom_measurements[i]
    x0[3*(i+1) + 2] = normalize_angle(x0[3*(i+1) + 2])

# ランドマークの初期値: 複数の観測から三角測量風に初期化
for j in range(n_landmarks):
    # このランドマークを観測した最初の2つのポーズから初期化
    obs_for_j = [(k, pi) for k, (pi, lj) in enumerate(visibility) if lj == j]
    if len(obs_for_j) >= 2:
        k1, pi1 = obs_for_j[0]
        k2, pi2 = obs_for_j[1]
        px1, py1, pth1 = get_pose(x0, pi1)
        px2, py2, pth2 = get_pose(x0, pi2)
        b1 = pth1 + bearing_measurements[k1]
        b2 = pth2 + bearing_measurements[k2]
        # 2本の光線の交点で初期化
        A_tri = np.array([[np.cos(b1), -np.cos(b2)],
                          [np.sin(b1), -np.sin(b2)]])
        b_tri = np.array([px2 - px1, py2 - py1])
        try:
            ts = np.linalg.solve(A_tri, b_tri)
            lx = px1 + ts[0] * np.cos(b1)
            ly = py1 + ts[0] * np.sin(b1)
            x0[lm_idx(j)] = [lx, ly]
        except np.linalg.LinAlgError:
            px, py, pth = get_pose(x0, obs_for_j[0][1])
            bearing = bearing_measurements[obs_for_j[0][0]]
            x0[lm_idx(j)] = [px + 4.0*np.cos(pth + bearing),
                             py + 4.0*np.sin(pth + bearing)]
    else:
        k, pi = obs_for_j[0]
        px, py, pth = get_pose(x0, pi)
        bearing = bearing_measurements[k]
        x0[lm_idx(j)] = [px + 4.0*np.cos(pth + bearing),
                         py + 4.0*np.sin(pth + bearing)]

# 初期推定を表示
print("=== 初期推定値（オドメトリ累積 + 三角測量初期化） ===")
for i in range(n_poses):
    p = get_pose(x0, i)
    p_true = poses_true[i]
    print(f"  p{i}: ({p[0]:6.3f}, {p[1]:6.3f}, {np.degrees(p[2]):6.1f}°)  "
          f"真値: ({p_true[0]:6.3f}, {p_true[1]:6.3f}, {np.degrees(p_true[2]):6.1f}°)")
for j in range(n_landmarks):
    l = get_landmark(x0, j)
    l_true = landmarks_true[j]
    print(f"  l{j}: ({l[0]:6.3f}, {l[1]:6.3f})  真値: ({l_true[0]:6.3f}, {l_true[1]:6.3f})")

In [ ]:
# =============================================================
# Gauss-Newton法を実行! (LMダンピング付きで安定化)
# =============================================================
x_est, history = gauss_newton_slam(x0, max_iter=50, lm_lambda=0.01)

print(f"\n=== 最終推定結果 ===")
print(f"{'':>6s}  {'推定値':>24s}  {'真値':>24s}  {'位置誤差':>8s}")
print("-" * 70)
for i in range(n_poses):
    p = get_pose(x_est, i)
    p_true = poses_true[i]
    err = np.linalg.norm(p[:2] - p_true[:2])
    print(f"  p{i}:  ({p[0]:7.4f}, {p[1]:7.4f}, {np.degrees(p[2]):7.2f}°)  "
          f"({p_true[0]:7.4f}, {p_true[1]:7.4f}, {np.degrees(p_true[2]):7.2f}°)  {err:.4f}m")
for j in range(n_landmarks):
    l = get_landmark(x_est, j)
    l_true = landmarks_true[j]
    err = np.linalg.norm(l - l_true)
    print(f"  l{j}:  ({l[0]:7.4f}, {l[1]:7.4f})              "
          f"({l_true[0]:7.4f}, {l_true[1]:7.4f})              {err:.4f}m")

## 2.3 結果の可視化

In [ ]:
# =============================================================
# 結果の可視化: 真値 vs 初期推定 vs 最終推定
# =============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax_idx, (ax, title) in enumerate(zip(axes, ['初期推定 vs 真値', '最終推定 (Gauss-Newton) vs 真値'])):
    x_plot = x0 if ax_idx == 0 else x_est
    
    # 真値
    ax.plot(poses_true[:, 0], poses_true[:, 1], 'k--', alpha=0.4, label='True trajectory')
    for i, (x, y, th) in enumerate(poses_true):
        ax.plot(x, y, 'ko', markersize=6, alpha=0.4)
        ax.arrow(x, y, 0.3*np.cos(th), 0.3*np.sin(th),
                 head_width=0.08, fc='gray', ec='gray', alpha=0.4)
    for j, (lx, ly) in enumerate(landmarks_true):
        ax.plot(lx, ly, 'k*', markersize=12, alpha=0.4)
    
    # 推定値
    poses_est = np.array([get_pose(x_plot, i) for i in range(n_poses)])
    ax.plot(poses_est[:, 0], poses_est[:, 1], 'b-o', markersize=8, label='Estimated poses')
    for i in range(n_poses):
        px, py, pth = poses_est[i]
        ax.arrow(px, py, 0.3*np.cos(pth), 0.3*np.sin(pth),
                 head_width=0.08, fc='blue', ec='blue')
        ax.text(px+0.15, py+0.2, f'p{i}', fontsize=9, color='blue')
    
    lms_est = np.array([get_landmark(x_plot, j) for j in range(n_landmarks)])
    ax.plot(lms_est[:, 0], lms_est[:, 1], 'r*', markersize=15, label='Estimated landmarks')
    for j in range(n_landmarks):
        ax.text(lms_est[j, 0]+0.15, lms_est[j, 1]+0.15, f'l{j}', fontsize=9, color='red')
    
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# 収束の可視化
# =============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

iters = range(1, len(history) + 1)
costs = [h[0] for h in history]
deltas = [h[1] for h in history]

ax1.semilogy(iters, costs, 'b-o', markersize=6)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Cost $J(\\mathbf{x})$')
ax1.set_title('コスト関数の減少', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.semilogy(iters, deltas, 'r-o', markersize=6)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('$\\|\\delta\\|$')
ax2.set_title('更新ステップの大きさ', fontweight='bold')
ax2.axhline(y=1e-6, color='gray', linestyle='--', alpha=0.5, label='tol = 1e-6')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"→ Gauss-Newton法は{len(history)}回の反復で収束")
print(f"  初期コスト: {costs[0]:.4f} → 最終コスト: {costs[-1]:.4f}")

## 2.4 Gauss-Newton vs Levenberg-Marquardt

**Levenberg-Marquardt法** (SLAM Handbook Eq. 1.38) は正規方程式にダンピング項を追加します：

$$(\mathbf{A}^\top\mathbf{A} + \lambda \, \text{diag}(\mathbf{A}^\top\mathbf{A})) \, \boldsymbol{\delta}^{\text{lm}} = \mathbf{A}^\top\mathbf{b}$$

- $\lambda = 0$: Gauss-Newton と同じ（高速だが不安定な場合あり）
- $\lambda$ が大きい: 最急降下法に近づく（遅いが安定）

初期値が悪い場合にLMの方がロバストに収束するか比較してみましょう。

In [ ]:
# =============================================================
# GN vs LM の比較（悪い初期値で）
# =============================================================

# 初期値にさらにノイズを追加して悪い初期値を作る
x0_bad = x0.copy()
x0_bad += np.random.normal(0, 0.3, n_state)
for i in range(n_poses):
    x0_bad[3*i + 2] = normalize_angle(x0_bad[3*i + 2])

print("=== Levenberg-Marquardt (λ=0.001, 弱ダンピング) ===")
x_gn, hist_gn = gauss_newton_slam(x0_bad, max_iter=50, lm_lambda=0.001)

print("\n=== Levenberg-Marquardt (λ=0.1, 強ダンピング) ===")
x_lm, hist_lm = gauss_newton_slam(x0_bad, max_iter=50, lm_lambda=0.1)

# 比較プロット
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(range(1, len(hist_gn)+1), [h[0] for h in hist_gn], 'b-o',
             markersize=4, label='LM (λ=0.001)')
ax1.semilogy(range(1, len(hist_lm)+1), [h[0] for h in hist_lm], 'r-s',
             markersize=4, label='LM (λ=0.1)')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Cost')
ax1.set_title('コスト収束の比較', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogy(range(1, len(hist_gn)+1), [h[1] for h in hist_gn], 'b-o',
             markersize=4, label='LM (λ=0.001)')
ax2.semilogy(range(1, len(hist_lm)+1), [h[1] for h in hist_lm], 'r-s',
             markersize=4, label='LM (λ=0.1)')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('$\\|\\delta\\|$')
ax2.set_title('ステップサイズの比較', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2.5 演習問題

### 演習1: Range観測
Bearing観測の代わりに **Range（距離）観測** $h(\mathbf{p}, \boldsymbol{\ell}) = \sqrt{(\ell_x - p_x)^2 + (\ell_y - p_y)^2}$ を実装してみましょう。ヤコビアンはどう変わりますか？

### 演習2: Range + Bearing
Range と Bearing の両方を使う場合を実装して、Bearingだけの場合と精度を比較してみましょう。

### 演習3: 初期値の影響
`x0_bad` のノイズの大きさを変えて、Gauss-Newton法が発散するケースを見つけてみましょう。そのとき LM法は収束しますか？

## まとめ

- 2Dの **Bearing観測は非線形** → Gauss-Newton法で反復的に線形化して解く
- ヤコビアンを解析的に導出して正規方程式を構築
- **Gauss-Newton法**: 初期値が良ければ高速（2次収束）
- **Levenberg-Marquardt法**: ダンピングにより安定だが遅い場合あり
- 実際のSLAM（Ch.7以降）ではこの基本構造の上にセンサ固有の処理が加わる

### 次のNotebook
→ `03_sparsity_and_elimination.ipynb`: スパース性、変数消去、Bayes Tree